# Module 9 · 多模態 1：圖文配對與 VLM 資料格式

> **本節定位｜全新（2026 多模態大模型基礎）**
> 2026 最重要的趨勢之一是 **多模態大模型 (VLM)**：能同時看圖與讀文。
> 它們的資料是「**圖–文配對**」與「**帶圖的對話 (chat)**」。
> 本節說明這兩種資料結構，並用 CLIP 體驗圖文對齊。

## 學習目標
- 理解 **圖文對比學習 (CLIP)** 的資料結構：`{image, text}` 配對。
- 用 CLIP 計算圖文相似度（同一向量空間的跨模態檢索）。
- 認識 **VLM 對話資料格式**：messages 中混合 image 與 text 內容。

## 0. 資料結構設計：兩種多模態資料

**(1) 圖文配對（對比學習，如 CLIP 預訓練）**
```python
{"image": <PIL.Image>, "text": "a dog running on the beach"}
```
模型學習把「配對的圖與文」在向量空間拉近、把「不配對的」推遠。

**(2) 帶圖的對話（VLM 指令微調，如 LLaVA / Qwen-VL）**
```json
{"messages": [
  {"role": "user", "content": [
      {"type": "image"},
      {"type": "text", "text": "這張圖裡有什麼？"}]},
  {"role": "assistant", "content": [
      {"type": "text", "text": "一隻狗在沙灘上奔跑。"}]}
]}
```
圖像會被視覺編碼器轉成一串「視覺 token」，和文字 token 一起餵進 LLM。

In [1]:
import numpy as np
from PIL import Image
img = Image.fromarray((np.random.RandomState(0).rand(224, 224, 3) * 255).astype(np.uint8))

## 1. CLIP：圖與文在同一向量空間

CLIP 同時有影像編碼器與文字編碼器，輸出可直接比較的嵌入。需 `multimodal` extra。

In [2]:
try:
    import torch
    from transformers import CLIPModel, CLIPProcessor
    clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    texts = ["a dog on the beach", "a bowl of salad", "a city skyline at night"]
    inp = proc(text=texts, images=img, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = clip(**inp)
    print(f"image_embeds: {tuple(out.image_embeds.shape)}  (1, D)")
    print(f"text_embeds : {tuple(out.text_embeds.shape)}  (num_texts, D) ← 同一向量空間")
    probs = out.logits_per_image.softmax(dim=1)[0]
    print("\n圖與各文字的匹配機率（合成圖僅示意）：")
    for tdesc, p in zip(texts, probs):
        print(f"  {p.item():.3f}  {tdesc}")
except Exception as e:
    print("（未能下載 CLIP，略過）", e)
    print("概念：image_embeds 與 text_embeds 在同一空間，可互相檢索（以圖找文/以文找圖）。")

（未能下載 CLIP，略過） No module named 'torch'
概念：image_embeds 與 text_embeds 在同一空間，可互相檢索（以圖找文/以文找圖）。


## 2. VLM 對話資料：用 processor 把圖+文一起前處理

多模態模型有自己的 processor，會同時處理影像（→視覺 token）與文字（→文字 token），
攤平成模型輸入。下面示範資料**結構**（不下載大型 VLM）。

In [3]:
vlm_sample = {
    "messages": [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": "描述這張圖片。"}]},
        {"role": "assistant", "content": [
            {"type": "text", "text": "這是一張示範圖片。"}]},
    ]
}
import json
print("VLM 對話資料結構（image 與 text 混合在 content 內）：")
print(json.dumps(vlm_sample, ensure_ascii=False, indent=2))
print("\n→ 訓練時：影像 → 視覺編碼器 → 視覺 token，與文字 token 串接後一起進 LLM。")

VLM 對話資料結構（image 與 text 混合在 content 內）：
{
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "image"
        },
        {
          "type": "text",
          "text": "描述這張圖片。"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "type": "text",
          "text": "這是一張示範圖片。"
        }
      ]
    }
  ]
}

→ 訓練時：影像 → 視覺編碼器 → 視覺 token，與文字 token 串接後一起進 LLM。


## 小結

| 資料類型 | 結構 | 用途 |
|:--|:--|:--|
| 圖文配對 | `{image, text}` | CLIP 對比學習、跨模態檢索 |
| 帶圖對話 | `messages`（image+text 混合） | VLM 指令微調（看圖回答） |

**跨模態的關鍵**：把不同模態都編碼到**可比較/可串接的向量/ token**，
這正是 2026 多模態大模型的共同設計。

**延伸**：多模態與生成式模型的訓練藍圖，見 **Module 11 · `06_generative_and_multimodal_blueprint`**。